# RAG validation — real embeddings + retrieval + generation

Run this notebook in a fresh Google Colab runtime. It validates dense embeddings, cosine retrieval, evidence-bearing prompt assembly, and a small causal-language-model generation path.

In [ ]:
import json, os, platform, sys
from datetime import datetime, timezone
print(json.dumps({'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version, 'os': platform.platform(), 'machine': platform.machine(), 'cuda_visible': os.environ.get('CUDA_VISIBLE_DEVICES')}, indent=2))

In [ ]:
!pip -q install -U sentence-transformers transformers accelerate

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

In [ ]:
CORPUS = [
    {'id': 'colab', 'title': 'Colab runtime', 'text': 'Google Colab runtimes are ephemeral. Important checkpoints and artifacts should be persisted to durable storage so work can resume after a runtime reset.'},
    {'id': 'rag', 'title': 'RAG workflow', 'text': 'A RAG workflow retrieves relevant document chunks before generation and supplies those chunks as evidence to the language model.'},
    {'id': 'evaluation', 'title': 'RAG evaluation', 'text': 'RAG evaluation should separate retrieval quality from generation quality and keep the evaluation set frozen during tuning.'}
]
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
GENERATION_MODEL = 'sshleifer/tiny-gpt2'
TOP_K = 2
print(json.dumps({'embedding_model': EMBEDDING_MODEL, 'generation_model': GENERATION_MODEL, 'corpus_revision': 'rag-validation-v1', 'top_k': TOP_K}, indent=2))

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
documents = [item['text'] for item in CORPUS]
document_embeddings = embedder.encode_document(documents, normalize_embeddings=True, convert_to_tensor=True)
query = 'How should work survive a Colab runtime reset?'
query_embedding = embedder.encode_query(query, normalize_embeddings=True, convert_to_tensor=True)
scores = document_embeddings @ query_embedding
top_scores, top_indices = torch.topk(scores, k=TOP_K)
retrieved = [{'id': CORPUS[int(index)]['id'], 'title': CORPUS[int(index)]['title'], 'score': float(score), 'text': CORPUS[int(index)]['text']} for score, index in zip(top_scores, top_indices)]
print(json.dumps(retrieved, indent=2))
assert retrieved[0]['id'] == 'colab', 'Expected Colab evidence to rank first'

In [ ]:
def build_prompt(question, passages):
    evidence = '\n'.join(f"[{p['id']}] {p['text']}" for p in passages)
    return f'Answer only from the evidence below. Cite evidence IDs. Say the evidence is insufficient when unsupported.\n\nQuestion: {question}\n\nEvidence:\n{evidence}\n\nAnswer:'
prompt = build_prompt(query, retrieved)
print(prompt)
assert '[colab]' in prompt

In [ ]:
generator = pipeline('text-generation', model=GENERATION_MODEL, device=0 if torch.cuda.is_available() else -1)
output = generator(prompt, max_new_tokens=40, do_sample=False, return_full_text=False)[0]['generated_text']
print(output)

In [ ]:
result = {'embedding_model': EMBEDDING_MODEL, 'generation_model': GENERATION_MODEL, 'retrieved_ids': [p['id'] for p in retrieved], 'retrieval_hit_at_k': float(retrieved[0]['id'] == 'colab'), 'generation_nonempty': bool(output.strip())}
print(json.dumps(result, indent=2))
assert result['retrieval_hit_at_k'] == 1.0
assert result['generation_nonempty'] is True
print('RAG Colab validation passed')